## Unilingual Model Exploration

This section explores unilingual models (Ensemble methods) that uses one model per language


---
Note that cross-validation process differs if we use a multi-lingual model or mono-lingual model:
- Multi-Lingual: Each fold should contain all the nodes with the same sentence_id and for all languages! (To avoid unbalance)
- Uni-Lingual: Each fold should contain all the the nodes with the same sentence_id. There are 2 ways to do this:

In [1]:
# EL CLASSICO: 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 

# Classical Scikit-Learn Imports
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier

# Advanced ML Imports
import optuna
from catboost import CatBoostClassifier, Pool
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
# eZAutoML Imports
#from ezautoml.model import eZAutoML
#from ezautoml.space.search_space import SearchSpace
#from ezautoml.evaluation.metric import MetricSet, Metric
#from ezautoml.evaluation.task import TaskType
#from ezautoml.optimization.optimizers.random_search import RandomSearchOptimizer

# Custom libraries and functions
from src.unilingual_ensemble import UnilingualEnsembleClassifier
from src.cross_validation import run_groupkfold_cv
from src.submission import generate_kaggle_submission

def evaluate_model(y_true_df, y_pred_df):
    # Assuming y_true_df and y_pred_df are DataFrames with 'root' and matching length
    if 'root' not in y_true_df.columns or 'root' not in y_pred_df.columns:
        print("Error: 'root' column missing in one of the dataframes.")
        return
    if len(y_true_df) != len(y_pred_df):
        print("Error: Length mismatch between true and predicted values.")
        print(f"  Length of true values: {len(y_true_df)}")
        print(f"  Length of predicted values: {len(y_pred_df)}")
        return
        
    y_true = y_true_df['root']
    y_pred = y_pred_df['root']
    correct_predictions = (y_true == y_pred).sum()
    total_predictions = len(y_true)
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0
    print(f"Number of correct predictions: {correct_predictions} / {total_predictions}")
    print(f"Evaluation accuracy: {accuracy:.4f}")
    return accuracy


# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")
train

/home/wtroiani/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,sentence_id,language,node,sentence_length,root,degree,degree_squared,degree_diff,local_degree_ratio,max_neighbor_degree,...,local_entropy,global_entropy,eccentricity,closeness_centrality,kcore_number,avg_shortest_path_length,pca_1,pca_2,pca_3,cluster
0,2,Japanese,14,23,0,-0.178144,-0.214265,0.400779,-0.064361,-0.862076,...,0.197384,-0.267930,1.134275,-0.527223,-0.373383,0.849250,-0.726772,-0.639728,1.854627,0
1,2,Japanese,8,23,0,-0.178144,-0.214265,0.400779,-0.064361,-0.862076,...,0.197384,-0.267930,0.749748,-0.458162,-0.373383,0.310212,-0.609886,-0.935976,1.284601,0
2,2,Japanese,4,23,0,-0.803671,-0.673585,-0.078100,-0.559448,-0.862076,...,-0.888409,-0.267930,1.518802,-0.597152,-0.373383,1.578536,-2.356987,1.134445,2.251312,0
3,2,Japanese,6,23,0,-0.178144,-0.214265,0.400779,-0.064361,-0.351293,...,-0.007529,-0.267930,1.134275,-0.534229,-0.373383,0.912666,-0.746906,-0.410513,1.723765,0
4,2,Japanese,2,23,0,0.447383,0.551268,1.039284,0.727776,-0.862076,...,0.764089,-0.267930,0.749748,-0.458162,-0.373383,0.310212,0.414012,-1.678863,1.672658,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
197474,995,Russian,2,19,0,-0.671981,-0.641352,-0.758611,-0.654991,-0.028694,...,-0.888409,0.075185,-0.221689,-0.186386,-0.020751,0.207622,-1.644817,1.379274,-0.403877,1
197475,995,Russian,14,19,0,-0.671981,-0.641352,-1.918001,-0.814808,1.207936,...,-0.888409,0.075185,0.243791,-0.093511,-0.020751,-0.164092,-1.243046,2.636507,-1.697236,1
197476,995,Russian,5,19,0,0.085235,-0.085333,-0.178916,-0.255446,1.207936,...,-0.034032,0.075185,0.243791,-0.066927,-0.020751,-0.257020,0.020820,0.354585,-0.749208,1
197477,995,Russian,16,19,0,-0.671981,-0.641352,-0.178916,-0.455220,-0.647010,...,-0.888409,0.075185,0.709271,-0.253477,-0.020751,0.532871,-1.748921,1.085061,0.956803,1


### Model 1: Random Forest Ensemble

In [ ]:
target_col = "root"
group_col = "sentence_id" 


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=RandomForestClassifier,
    base_model_kwargs={'n_estimators': 100, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'n_estimators': [250, 500],            # Number of trees
    'max_depth': [None, 10, 20],           # Tree depth
    'class_weight': ['balanced']         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()

# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,
    X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_rf_unilingual.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


## Model 2: XGBoost Ensemble

In [ ]:
target_col = "root"
group_col = "sentence_id" 

# Run GroupKFold CV on train
cv_scores = run_groupkfold_cv(
    X=train,
    y=train[target_col],
    group_colname=group_col,
    clf_cls=UnilingualEnsembleClassifier,
    clf_kwargs={
        'base_model_cls': XGBClassifier,
        'base_model_kwargs': {'n_estimators': 300, 'n_jobs': 1},
        'language_colname': 'language',
        'n_jobs': 8  # adjust based on your CPU
    },
    n_splits=5,
    metric_fn=accuracy_score,
    verbose=True
)

# Plot CV scores
plt.plot(cv_scores, marker='o')
plt.title('CV Accuracy Scores per Fold')
plt.xlabel('Fold')
plt.ylabel('Accuracy')
plt.grid(True)
plt.show()

# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=XGBClassifier,
    base_model_kwargs={'n_estimators': 300, 'n_jobs': 1},
    language_colname='language',
    n_jobs=8
)
model.fit(train.drop(columns=target_col), train[target_col])

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_xgboost.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


## Model 3: LightGBM Ensemble

In [ ]:
target_col = "root"
group_col = "sentence_id" 


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=LGBMClassifier,
    base_model_kwargs={'n_estimators': 500, 'n_jobs': 1},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
    'max_depth': [None, 5, 10, 20],           # Tree depth; None allows full growth
    'class_weight': [None, 'balanced'],         # For imbalanced classes
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

[LightGBM] [Info] Number of positive: 250, number of negative: 3705
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4372
[LightGBM] [Info] Number of data points in the train set: 3955, number of used features: 25
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.063211 -> initscore=-2.695978
[LightGBM] [Info] Start training from score -2.695978
[LightGBM] [Info] Number of positive: 250, number of negative: 4063
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000389 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4497
[LightGBM] [Info] Number of data points in the train set: 4313, number of used features: 25
[LightGBM] [Info] [binary:

UnilingualEnsembleClassifier(base_model_cls=<class 'lightgbm.sklearn.LGBMClassifier'>,
                             base_model_kwargs={'n_estimators': 500,
                                                'n_jobs': 1},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'class_weight': [None, 'balanced'],
                                         'max_depth': [None, 5, 10, 20]})

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_lightgbm.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 13:45:28.580 | INFO     | src.submission:generate_kaggle_submission:29 - Generating predictions...
2025-05-29 13:45:38.604 | INFO     | src.submission:generate_kaggle_submission:42 - Detected multilingual (per-language model ensemble) setup.
2025-05-29 13:45:38.649 | SUCCESS  | src.submission:generate_kaggle_submission:75 - Submission saved to: data/predictions_submission_unilingual_lightgbm.csv


In [ ]:
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_lightgbm.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 2540 / 10395
Evaluation accuracy: 0.2443


## Model 4: SVM

In [ ]:
target_col = "root"
group_col = "sentence_id"


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=SVC,
    base_model_kwargs={"probability": True, "class_weight": "balanced"},
    language_colname='language',
    gridsearch_per_language=True,
    param_grid = {
        "C": [0.1, 1, 10],                     # Regularization strength
        "kernel": ["rbf"],          # Simpler and widely useful kernels
        "gamma": ["scale", "auto"],
    },
    cv=2,
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

2025-05-31 08:42:37.094 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Finnish
2025-05-31 08:43:15.423 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Czech
2025-05-31 08:43:34.584 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Turkish
2025-05-31 08:43:47.941 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Swedish
2025-05-31 08:43:51.517 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Icelandic
2025-05-31 08:44:06.104 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Polish
2025-05-31 08:44:12.879 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Russian
2025-05-31 08:44:20.143 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Korean
2025-05-31 08:44:33.162 | SUCCESS 

UnilingualEnsembleClassifier(base_model_cls=<class 'sklearn.svm._classes.SVC'>,
                             base_model_kwargs={'class_weight': 'balanced',
                                                'probability': True},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'C': [0.1, 1, 10],
                                         'gamma': ['scale', 'auto'],
                                         'kernel': ['rbf']})

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_svm_tuned.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-31 08:46:38.327 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...


2025-05-31 08:46:50.666 | INFO     | src.submission:generate_kaggle_submission:55 - Detected multilingual (per-language model ensemble) setup.
2025-05-31 08:46:50.693 | SUCCESS  | src.submission:generate_kaggle_submission:88 - Submission saved to: data/predictions_submission_unilingual_svm_tuned.csv


In [ ]:
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_svm_tuned.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

An error occurred during custom evaluation: name 'pd' is not defined


### Model 5: eZAutoML Exploration

In [ ]:
target_col = "root"
group_col = "sentence_id"  

# === Extract features and target ===
X = train.drop(columns=[target_col])
y = train[target_col]

# === Train/test split (optional if using full train set) ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# === Define metrics ===
metrics = MetricSet(
    {"accuracy": Metric(name="accuracy", fn=accuracy_score, minimize=False)},
    primary_metric_name="accuracy"
)

# === Define search space ===
search_space = SearchSpace.from_builtin("classification_space")

# === Initialize eZAutoML ===
ezautoml = eZAutoML(
    search_space=search_space,
    task=TaskType.CLASSIFICATION,
    metrics=metrics,
    max_trials=50,     # You can increase for better models
    max_time=600,      # 10 minutes
    seed=42
)

# === Fit the AutoML model ===
ezautoml.fit(X_train, y_train)

# === Evaluate ===
test_accuracy = ezautoml.test(X_test, y_test)
print("Test accuracy:", test_accuracy)

# === Summary of best models ===
ezautoml.summary(k=50)


ModuleNotFoundError: No module named 'torch'

## Model 6: HistogramBoosting (UNI)

In [ ]:
target_col = "root"
group_col = "sentence_id" 


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=HistGradientBoostingClassifier,
    base_model_kwargs={"early_stopping": True},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid = {
        'max_depth': [None, 6, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_iter': [100, 300, 500],
        'l2_regularization': [0.01, 0.1, 1],
        'min_samples_leaf': [20, 50, 100],
    },
    n_jobs=21
)
model.fit(train.drop(columns=target_col), train[target_col])

UnilingualEnsembleClassifier(base_model_cls=<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>,
                             base_model_kwargs={'early_stopping': True},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'l2_regularization': [0.01, 0.1, 1],
                                         'learning_rate': [0.01, 0.05, 0.1],
                                         'max_depth': [None, 6, 10],
                                         'max_iter': [100, 300, 500],
                                         'min_samples_leaf': [20, 50, 100]})

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_hist.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)


2025-05-29 20:05:31.560 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...
2025-05-29 20:05:34.423 | INFO     | src.submission:generate_kaggle_submission:55 - Detected multilingual (per-language model ensemble) setup.
2025-05-29 20:05:34.437 | SUCCESS  | src.submission:generate_kaggle_submission:88 - Submission saved to: data/predictions_submission_unilingual_hist.csv


In [ ]:
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_hist.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

Number of correct predictions: 3055 / 10395
Evaluation accuracy: 0.2939


### 7. AdaBoosting (UNI)

In [ ]:
target_col = "root"
group_col = "sentence_id"


# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=AdaBoostClassifier,
    base_model_kwargs={"estimator": DecisionTreeClassifier(), "n_estimators": 300},
    language_colname='language',
    gridsearch_per_language=True,
    cv=2,
    param_grid={
        'learning_rate': [0.01, 0.1, 1.0],
        'estimator__max_depth': [None, 10],
    },
    n_jobs=21
)

model.fit(train.drop(columns=target_col), train[target_col])

2025-05-31 11:12:27.542 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Turkish
2025-05-31 11:12:32.086 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Finnish
2025-05-31 11:12:54.863 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Korean
2025-05-31 11:13:22.455 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Indonesian
2025-05-31 11:13:29.131 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Czech
2025-05-31 11:13:30.227 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Icelandic
2025-05-31 11:13:58.421 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Swedish
2025-05-31 11:14:03.078 | SUCCESS  | src.unilingual_ensemble:_fit_one:53 - Finished training model for language: Polish
2025-05-31 11:14:13.288 | SUCCE

UnilingualEnsembleClassifier(base_model_cls=<class 'sklearn.ensemble._weight_boosting.AdaBoostClassifier'>,
                             base_model_kwargs={'estimator': DecisionTreeClassifier(),
                                                'n_estimators': 300},
                             gridsearch_per_language=True, n_jobs=21,
                             param_grid={'estimator__max_depth': [None, 10],
                                         'learning_rate': [0.01, 0.1, 1.0]})

In [4]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_adabooster.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_adabooster.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

2025-05-31 11:15:39.734 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...
2025-05-31 11:16:13.666 | INFO     | src.submission:generate_kaggle_submission:55 - Detected multilingual (per-language model ensemble) setup.
2025-05-31 11:16:13.691 | SUCCESS  | src.submission:generate_kaggle_submission:88 - Submission saved to: data/predictions_submission_unilingual_adabooster.csv


Number of correct predictions: 2589 / 10395
Evaluation accuracy: 0.2491


### 8. Catboost (UNI)

In [ ]:
# Load data
train = pd.read_csv("./data/train_dataset_ultraprocessed.csv")
test = pd.read_csv("./data/test_dataset_ultraprocessed.csv")

# Make sure 'root' is your target and 'sentence_id' groups sentences (adjust if needed)
target_col = "root"
group_col = "sentence_id"  # replace if different in your dataset

# Train final model on full train data
model = UnilingualEnsembleClassifier(
    base_model_cls=CatBoostClassifier,
    base_model_kwargs={"auto_class_weights": "Balanced", "depth": 14},
    language_colname='language',
    gridsearch_per_language=False,
    cv=2,
    param_grid = {
        'depth': [4, 6],
        'learning_rate': [0.03, 0.1]
    },
    n_jobs=1 # Do not put more if you dont want your laptop to mfking die 
    # Apparently catboost uses a weird cat_boost_info folder, therefore we cannot train them in parallel god knows why. I hate microsoft
)

model.fit(train.drop(columns=target_col), train[target_col])

Learning rate set to 0.026628
0:	learn: 0.6765894	total: 330ms	remaining: 5m 30s
1:	learn: 0.6614842	total: 384ms	remaining: 3m 11s
2:	learn: 0.6487818	total: 587ms	remaining: 3m 15s
3:	learn: 0.6372568	total: 786ms	remaining: 3m 15s
4:	learn: 0.6198561	total: 997ms	remaining: 3m 18s
5:	learn: 0.6074655	total: 1.19s	remaining: 3m 17s
6:	learn: 0.5955286	total: 1.4s	remaining: 3m 19s
7:	learn: 0.5831774	total: 1.6s	remaining: 3m 18s
8:	learn: 0.5701153	total: 1.79s	remaining: 3m 17s
9:	learn: 0.5602259	total: 1.98s	remaining: 3m 15s
10:	learn: 0.5516723	total: 2.18s	remaining: 3m 16s
11:	learn: 0.5414230	total: 2.37s	remaining: 3m 15s
12:	learn: 0.5309616	total: 2.58s	remaining: 3m 15s
13:	learn: 0.5237470	total: 2.77s	remaining: 3m 14s
14:	learn: 0.5136018	total: 2.96s	remaining: 3m 14s
15:	learn: 0.5015518	total: 3.16s	remaining: 3m 14s
16:	learn: 0.4932836	total: 3.35s	remaining: 3m 14s
17:	learn: 0.4858704	total: 3.55s	remaining: 3m 13s
18:	learn: 0.4789503	total: 3.74s	remaining: 3

In [ ]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_catboost.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_catboost.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

### 9. Catboost (MULTI)

#### Optuna Bayesian TPE Optimization

In [ ]:
# Define target and features
target_col = "root"
X = train.drop(columns=[target_col])
y = train[target_col]

# Identify categorical columns (assuming object dtype = categorical)
categorical_cols = X.select_dtypes(include='object').columns.tolist()

def objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 6, 16),  # deeper trees for complex patterns
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),  # fine steps but not too slow
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 15),  # regularization to avoid overfitting
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),  # controls randomness, helps generalize
        'border_count': trial.suggest_int('border_count', 16, 256),  # number of splits for numeric features (default 254, lower can speed up)
        'iterations': 500,  # enough rounds to learn without too long training
        'random_seed': 42,
        'auto_class_weights': 'Balanced'
    }

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    model = CatBoostClassifier(**params, verbose=0)
    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        cat_features=categorical_cols,
        early_stopping_rounds=30
    )

    preds = model.predict(X_valid)

    y_valid_df = pd.DataFrame({'root': y_valid})
    y_pred_df = pd.DataFrame({'root': preds}, index=y_valid.index)  # fix here!

    accuracy = evaluate_model(y_valid_df, y_pred_df)
    return accuracy



# Run Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("Best params:", study.best_params)


[I 2025-05-31 11:54:31,432] A new study created in memory with name: no-name-96faa332-2bda-468c-81d2-5a4e26e5b2f9


[I 2025-05-31 11:54:53,307] Trial 0 finished with value: 0.7710654243467693 and parameters: {'depth': 11, 'learning_rate': 0.03275805265177091, 'l2_leaf_reg': 4.787462588970133, 'bagging_temperature': 0.403109766896109, 'border_count': 125}. Best is trial 0 with value: 0.7710654243467693.


Number of correct predictions: 30454 / 39496
Evaluation accuracy: 0.7711


[I 2025-05-31 11:56:00,550] Trial 1 finished with value: 0.781193032205793 and parameters: {'depth': 16, 'learning_rate': 0.1118929507790916, 'l2_leaf_reg': 1.8897856807830096, 'bagging_temperature': 0.28525510488145056, 'border_count': 114}. Best is trial 1 with value: 0.781193032205793.


Number of correct predictions: 30854 / 39496
Evaluation accuracy: 0.7812


...

Number of correct predictions: 30732 / 39496
Evaluation accuracy: 0.7781


[I 2025-05-31 13:06:07,324] Trial 45 finished with value: 0.7904344743771521 and parameters: {'depth': 15, 'learning_rate': 0.019230656191278322, 'l2_leaf_reg': 2.416599811267049, 'bagging_temperature': 0.860310245189181, 'border_count': 36}. Best is trial 25 with value: 0.7953463641887786.


Number of correct predictions: 31219 / 39496
Evaluation accuracy: 0.7904


[I 2025-05-31 13:06:32,624] Trial 46 finished with value: 0.7740277496455338 and parameters: {'depth': 13, 'learning_rate': 0.03312822100857467, 'l2_leaf_reg': 3.408251769499449, 'bagging_temperature': 0.7316375144118288, 'border_count': 64}. Best is trial 25 with value: 0.7953463641887786.


Number of correct predictions: 30571 / 39496
Evaluation accuracy: 0.7740


In [2]:
target_col = "root"
categorical_cols = ["language"] 

#best_params = study.best_params
best_params = {'depth': 16, 'learning_rate': 0.01268179364871682, 'l2_leaf_reg': 2.0414082304544436, 'bagging_temperature': 0.9426379204160849, 'border_count': 110}

# Define Pool for training
train_pool = Pool(
    data=train.drop(columns=target_col),
    label=train[target_col],
    cat_features=categorical_cols
)

# Define the CatBoost model
model = CatBoostClassifier(**best_params, early_stopping_rounds=300)
# Train the model
model.fit(train_pool)


0:	learn: 0.6711502	total: 1.63s	remaining: 27m 9s
1:	learn: 0.6502157	total: 3.03s	remaining: 25m 12s
2:	learn: 0.6302972	total: 4.68s	remaining: 25m 55s
3:	learn: 0.6112316	total: 5.34s	remaining: 22m 10s
4:	learn: 0.5932740	total: 7.14s	remaining: 23m 40s
5:	learn: 0.5776756	total: 7.17s	remaining: 19m 48s
6:	learn: 0.5614763	total: 7.23s	remaining: 17m 5s
7:	learn: 0.5465926	total: 7.28s	remaining: 15m 2s
8:	learn: 0.5312217	total: 8.7s	remaining: 15m 57s
9:	learn: 0.5165175	total: 10.3s	remaining: 17m 4s
10:	learn: 0.5040399	total: 10.4s	remaining: 15m 35s
11:	learn: 0.4911306	total: 10.5s	remaining: 14m 23s
12:	learn: 0.4805973	total: 10.5s	remaining: 13m 18s
13:	learn: 0.4677525	total: 12.6s	remaining: 14m 50s
14:	learn: 0.4556227	total: 12.8s	remaining: 14m 1s
15:	learn: 0.4442183	total: 14.9s	remaining: 15m 15s
16:	learn: 0.4327909	total: 16.7s	remaining: 16m 2s
17:	learn: 0.4227366	total: 17.5s	remaining: 15m 56s
18:	learn: 0.4126091	total: 19.4s	remaining: 16m 39s
19:	learn:

In [3]:
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_multilingual_catboost.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_multilingual_catboost.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

2025-05-31 15:36:13.922 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...
2025-05-31 15:36:14.331 | INFO     | src.submission:generate_kaggle_submission:55 - Detected multilingual (per-language model ensemble) setup.
2025-05-31 15:36:14.354 | SUCCESS  | src.submission:generate_kaggle_submission:88 - Submission saved to: data/predictions_submission_multilingual_catboost.csv


Number of correct predictions: 3016 / 10395
Evaluation accuracy: 0.2901


### 10 Multi-Layer Perceptron

In [ ]:
target_col = "root"
categorical_cols = ["language"]

model = UnilingualEnsembleClassifier(
    base_model_cls=MLPClassifier,
    base_model_kwargs = {
        'hidden_layer_sizes': (2048, 1024, 512, 256, 128),  # a fairly large MLP
        'activation': 'relu',
        'learning_rate_init': 0.001,
        'alpha': 0.15, # L2 regularization strenght (This is ok) 
        'max_iter': 200,
        'early_stopping': True,
        'solver': 'adam',
        'batch_size': 'auto',
        'random_state': 42,
    },
    language_colname='language',
    gridsearch_per_language=False,  # Enable grid search only on learning rate
    cv=2,
    param_grid={
        'learning_rate_init': [0.0001, 0.001, 0.01]
    },
    n_jobs=8
)

model.fit(train.drop(columns=target_col), train[target_col])
joblib.dump(model, "./data/fat_mlp.pkl")

NameError: name 'UnilingualEnsembleClassifier' is not defined

In [ ]:
loaded_model = joblib.load('fat_mlp.pkl')
# 1. Prepare metadata from test
test_meta = test[['sentence_id', 'node', 'language']].copy()
# 2. Remove target column from test if exists
X_test = test.drop(columns=[target_col]) if target_col in test.columns else test

# 3. Generate submission
generate_kaggle_submission(
    model=model,X_test=X_test,
    test_meta=test_meta,
    output_path="data/predictions_submission_unilingual_mlp.csv",
    language_prefix="language_",  # Only relevant if X_test contains one-hot language columns
    y_true=test[target_col] if target_col in test.columns else None,
    return_df=False,
    verbose=True
)
try:
    # Reload for this function to ensure clean state
    kaggle_perfect_predictions_eval = pd.read_csv("data/kaggle_perfect_predictions.csv")
    current_predictions_eval = pd.read_csv('data/predictions_submission_unilingual_mlp.csv')
    evaluate_model(kaggle_perfect_predictions_eval, current_predictions_eval)
except FileNotFoundError:
    print("One of the prediction files not found. Skipping custom evaluation.")
except Exception as e:
    print(f"An error occurred during custom evaluation: {e}")

2025-05-31 17:27:43.004 | INFO     | src.submission:generate_kaggle_submission:42 - Generating predictions...
